In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import pandas as pd
df = pd.read_csv('/content/drive/MyDrive/multimodal_emotion/data/labels.csv')
print(df.shape)
print(df['label'].value_counts())

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
(4869, 4)
label
neutral     1921
positive    1731
negative    1217
Name: count, dtype: int64


In [ ]:
import torch
import torch.nn as nn
import numpy as np
from torchvision import models, transforms
from torch.utils.data import Dataset, DataLoader
from PIL import Image
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, f1_score
from sklearn.preprocessing import LabelEncoder
import json, os

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Usando: {device}")

Usando: cuda


In [ ]:
# Las rutas del CSV apuntan a data/raw/data/X.jpg (ruta local Mac)
# En Colab necesitamos redirigirlas a Drive

DRIVE_IMG_DIR = '/content/drive/MyDrive/multimodal_emotion/data/raw/data'

def fix_path(local_path):
    filename = os.path.basename(local_path)
    return os.path.join(DRIVE_IMG_DIR, filename)

df['image_path'] = df['image_path'].apply(fix_path)

# Verificar que las imágenes existen
existing = df['image_path'].apply(os.path.exists)
print(f"Imágenes encontradas: {existing.sum()} / {len(df)}")
df = df[existing].reset_index(drop=True)
print(f"Dataset final: {df.shape}")

Imágenes encontradas: 4869 / 4869
Dataset final: (4869, 4)


In [ ]:
TRAIN_TF = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.ColorJitter(0.2, 0.2, 0.1),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406],
                         [0.229, 0.224, 0.225])
])

VAL_TF = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406],
                         [0.229, 0.224, 0.225])
])

class ImageDataset(Dataset):
    def __init__(self, paths, labels, transform):
        self.paths     = paths
        self.labels    = labels
        self.transform = transform

    def __len__(self):
        return len(self.paths)

    def __getitem__(self, idx):
        img = Image.open(self.paths[idx]).convert('RGB')
        return self.transform(img), self.labels[idx]

In [ ]:
le     = LabelEncoder()
labels = le.fit_transform(df['label'].tolist())
paths  = df['image_path'].tolist()
n_cls  = len(le.classes_)
print(f"Clases: {le.classes_}")

tr_p, te_p, tr_l, te_l = train_test_split(
    paths, labels, test_size=0.2, random_state=42, stratify=labels)

tr_p, va_p, tr_l, va_l = train_test_split(
    tr_p, tr_l, test_size=0.125, random_state=42)

tr_dl = DataLoader(ImageDataset(tr_p, tr_l, TRAIN_TF),
                   batch_size=32, shuffle=True, num_workers=2)
va_dl = DataLoader(ImageDataset(va_p, va_l, VAL_TF),
                   batch_size=32, num_workers=2)
te_dl = DataLoader(ImageDataset(te_p, te_l, VAL_TF),
                   batch_size=32, num_workers=2)

print(f"Train: {len(tr_p)} | Val: {len(va_p)} | Test: {len(te_p)}")

Clases: ['negative' 'neutral' 'positive']
Train: 3408 | Val: 487 | Test: 974


In [ ]:
def build_resnet(num_classes):
    model = models.resnet18(weights='IMAGENET1K_V1')
    for p in model.parameters():
        p.requires_grad = False
    model.fc = nn.Linear(model.fc.in_features, num_classes)
    return model

model = build_resnet(n_cls).to(device)
crit  = nn.CrossEntropyLoss()
opt   = torch.optim.Adam(model.fc.parameters(), lr=2e-4)
sched = torch.optim.lr_scheduler.StepLR(opt, step_size=4, gamma=0.5)
print("Modelo listo")

Modelo listo


In [ ]:
train_losses = []

for epoch in range(10):
    model.train()
    total = 0
    for imgs, lbls in tr_dl:
        opt.zero_grad()
        loss = crit(model(imgs.to(device)), lbls.to(device))
        loss.backward()
        opt.step()
        total += loss.item()
    sched.step()
    avg = total / len(tr_dl)
    train_losses.append(avg)
    print(f"Epoch {epoch+1}/10 — Loss: {avg:.4f}")

Epoch 1/10 — Loss: 1.1477
Epoch 2/10 — Loss: 1.1094
Epoch 3/10 — Loss: 1.0882
Epoch 4/10 — Loss: 1.0724
Epoch 5/10 — Loss: 1.0657
Epoch 6/10 — Loss: 1.0568
Epoch 7/10 — Loss: 1.0599
Epoch 8/10 — Loss: 1.0529
Epoch 9/10 — Loss: 1.0483
Epoch 10/10 — Loss: 1.0456


In [ ]:
model.eval()
all_preds, all_probas = [], []

with torch.no_grad():
    for imgs, _ in te_dl:
        probs = torch.softmax(model(imgs.to(device)), dim=1).cpu().numpy()
        all_probas.extend(probs.tolist())
        all_preds.extend(np.argmax(probs, axis=1).tolist())

print("=== ResNet18 Transfer Learning ===")
print(classification_report(te_l, all_preds, target_names=le.classes_))
print(f"F1 macro: {f1_score(te_l, all_preds, average='macro'):.4f}")

=== ResNet18 Transfer Learning ===
              precision    recall  f1-score   support

    negative       0.42      0.22      0.29       244
     neutral       0.41      0.54      0.47       384
    positive       0.41      0.42      0.41       346

    accuracy                           0.41       974
   macro avg       0.42      0.39      0.39       974
weighted avg       0.42      0.41      0.40       974

F1 macro: 0.3899


In [ ]:
os.makedirs('/content/drive/MyDrive/multimodal_emotion/results', exist_ok=True)

results = {
    'f1_macro':      f1_score(te_l, all_preds, average='macro'),
    'probas':        all_probas,
    'train_losses':  train_losses,
    'label_encoder': le.classes_.tolist()
}

with open('/content/drive/MyDrive/multimodal_emotion/results/metrics_resnet.json', 'w') as f:
    json.dump(results, f, indent=2)

print("Guardado en Drive")

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(7, 4))
plt.plot(range(1, 11), train_losses, marker='o', color='#ea580c', linewidth=2)
plt.title('Curva de pérdida — ResNet18 Transfer Learning')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.xticks(range(1, 11))
plt.tight_layout()
plt.savefig('/content/drive/MyDrive/multimodal_emotion/results/resnet_loss_curve.png', dpi=150)
plt.show()

In [ ]:
import os

DRIVE_IMG_DIR = '/content/drive/MyDrive/multimodal_emotion/data/raw/data'

# Ver si la carpeta existe
print("¿Existe la carpeta?", os.path.exists(DRIVE_IMG_DIR))

# Contar archivos dentro
if os.path.exists(DRIVE_IMG_DIR):
    files = os.listdir(DRIVE_IMG_DIR)
    print(f"Archivos en la carpeta: {len(files)}")
    print("Primeros 5:", files[:5])
else:
    # Buscar dónde están realmente las imágenes
    base = '/content/drive/MyDrive/multimodal_emotion'
    for root, dirs, files in os.walk(base):
        if any(f.endswith('.jpg') for f in files):
            print(f"Imágenes encontradas en: {root}")
            print(f"Cantidad: {len([f for f in files if f.endswith('.jpg')])}")
            break

¿Existe la carpeta? True
Archivos en la carpeta: 9965
Primeros 5: ['4628.jpg', '4641.txt', '4683.txt', '4672.txt', '4675.txt']
